# Machine Learning Debugging, Gradients & Evaluation

## Milestone 1: Debug a Silently Failing Training Run

### 1. Identified Bugs & Symptoms
1. **`np.random.shuffle(X_tr)` without `y_tr`:** Shuffles features while leaving labels in their original order. The model memorizes random noise (falling training loss) but learns zero real signal (validation accuracy stuck near 50-55%).
2. **`StandardScaler().fit_transform(X)` before splitting:** Leaks validation distribution statistics (mean and variance) into the training set.
3. **`scheduler.step()` inside the batch loop:** Decays the learning rate by `0.9` after *every batch* instead of every epoch. Within a few epochs, learning rate hits $0$ and weights freeze.
4. **Missing `model.eval()` and `torch.no_grad()` during validation:** `Dropout(0.5)` stays active and `BatchNorm1d` updates running stats on validation data, causing validation accuracy to fluctuate randomly between epochs.
5. **Tensor dtype mismatch:** Passing `torch.tensor(X_val)` defaults to `float64` if `X` is a NumPy float array, whereas PyTorch layers expect `float32`.
6. **Sequential Slicing Split (`X[:8000]`):** Slicing data sequentially by index without a random stratified split (`train_test_split(..., stratify=y)`) risks severe class or temporal imbalance if the raw dataset is ordered.

### 2. Black-Box Diagnostics (Without Code Access)
- **Single-Batch Overfit Test:** Train on 1 batch of 32 samples. If it fails to converge quickly or validation fluctuates on the exact same batch, `Dropout`/`BatchNorm` is stuck in train mode or LR is decaying per step.
- **Label Permutation Test:** Train on intentionally shuffled labels. If the loss curve looks identical to the normal run, features and labels are misaligned in the data pipeline.
- **Repeated Evaluation Check:** Run inference twice on the validation set without training in between. If predictions change, stochastic layers (`Dropout`) are active during evaluation.

In [1]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# Dummy data for verification
X = np.random.randn(1000, 40)
y = np.random.randint(0, 2, size=1000)

# 1. Split BEFORE scaling to prevent data leakage
X_tr, X_val, y_tr, y_val = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

scaler = StandardScaler()
X_tr = scaler.fit_transform(X_tr)
X_val = scaler.transform(X_val)

# 2. Shuffle X and y together via DataLoader
train_ds = TensorDataset(torch.tensor(X_tr, dtype=torch.float32),
                         torch.tensor(y_tr, dtype=torch.long))
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)

model = nn.Sequential(
    nn.Linear(40, 128), nn.BatchNorm1d(128), nn.ReLU(),
    nn.Dropout(0.5), nn.Linear(128, 2)
)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.StepLR(opt, step_size=1, gamma=0.95)
loss_fn = nn.CrossEntropyLoss()

for epoch in range(5):
    model.train()
    for xb, yb in train_loader:
        opt.zero_grad()
        loss = loss_fn(model(xb), yb)
        loss.backward()
        opt.step()

    # 3. Step scheduler once per epoch
    scheduler.step()

    # 4. Proper evaluation mode
    model.eval()
    with torch.no_grad():
        val_logits = model(torch.tensor(X_val, dtype=torch.float32))
        val_preds = val_logits.argmax(dim=1).numpy()
        val_acc = (val_preds == y_val).mean()
    print(f"Epoch {epoch+1} | Val Acc: {val_acc:.2%}")

Epoch 1 | Val Acc: 45.00%
Epoch 2 | Val Acc: 47.50%
Epoch 3 | Val Acc: 47.00%
Epoch 4 | Val Acc: 49.00%
Epoch 5 | Val Acc: 47.50%


## Milestone 2: Single-Head Attention Gradients

### 1. Backward Pass Derivation
Given forward pass:
$$S = \frac{QK^T}{\sqrt{d}}, \quad P = \text{softmax}(S), \quad O = PV$$

For upstream gradient $dO \in \mathbb{R}^{n \times d_v}$:
1. **Gradient w.r.t $V$ and $P$:**
   $$dV = P^T dO, \quad dP = dO \, V^T$$
2. **Gradient through Row-wise Softmax ($dS$):**
   For each row $i$, the Jacobian of softmax gives:
   $$dS = P \odot \left( dP - \text{rowsum}(dP \odot P) \right)$$
3. **Gradient w.r.t $Q$ and $K$:**
   $$dQ = \frac{dS \, K}{\sqrt{d}}, \quad dK = \frac{dS^T Q}{\sqrt{d}}$$

### 2. Theory & Stability Questions
- **Forward Numerical Stability:** Large values in $S$ cause $e^{S_{ij}}$ to overflow (`inf`). **Fix:** Subtract the row-wise maximum before exponentiating: $S' = S - \max_j(S_{ij})$.
- **Why scale by $1/\sqrt{d}$:** If $q, k$ have zero mean and unit variance, their dot product has variance $d$. For large $d$, values grow large, pushing softmax into flat saturation regions where gradients vanish ($\approx 0$). Dividing by $\sqrt{d}$ restores unit variance.
- **Causal Mask in Backward Pass:** In the forward pass, masked positions ($j > i$) are set to $-\infty$ before softmax, resulting in $P_{ij} = 0$. Since $P_{ij} = 0$, the softmax gradient formula naturally sets $dS_{ij} = 0$ for all masked positions without extra masking code.
- **Memory Cost & FlashAttention:** Standard attention stores $S$ and $P$ of size $O(n^2)$ in GPU High Bandwidth Memory (HBM). **FlashAttention** splits $Q, K, V$ into blocks in fast SRAM and uses online softmax rescaling to compute $O$ directly without ever materializing the $n \times n$ matrix in HBM, reducing memory to $O(n)$.

In [2]:
# Milestone 2: NumPy Forward/Backward & Numerical Gradient Check
np.random.seed(42)

def attention_forward(Q, K, V, causal=False):
    d = Q.shape[-1]
    S = (Q @ K.T) / np.sqrt(d)
    if causal:
        mask = np.triu(np.ones_like(S, dtype=bool), k=1)
        S = np.where(mask, -1e9, S)

    # Numerically stable softmax
    S_max = np.max(S, axis=-1, keepdims=True)
    exp_S = np.exp(S - S_max)
    P = exp_S / np.sum(exp_S, axis=-1, keepdims=True)
    O = P @ V
    return O, (Q, K, V, P)

def attention_backward(dO, cache):
    Q, K, V, P = cache
    d = Q.shape[-1]

    dV = P.T @ dO
    dP = dO @ V.T

    # Row-wise softmax backward
    row_dot = np.sum(dP * P, axis=-1, keepdims=True)
    dS = P * (dP - row_dot)

    dQ = (dS @ K) / np.sqrt(d)
    dK = (dS.T @ Q) / np.sqrt(d)
    return dQ, dK, dV

# Numerical Gradient Check (float64, eps = 1e-6, threshold < 1e-6)
n, d = 6, 8
Q = np.random.randn(n, d).astype(np.float64)
K = np.random.randn(n, d).astype(np.float64)
V = np.random.randn(n, d).astype(np.float64)
dO = np.random.randn(n, d).astype(np.float64)

O, cache = attention_forward(Q, K, V, causal=True)
dQ, dK, dV = attention_backward(dO, cache)

def rel_error(a, b):
    return np.max(np.abs(a - b) / np.maximum(1e-8, np.abs(a) + np.abs(b)))

eps = 1e-6
for name, param, analytic_grad in [("dQ", Q, dQ), ("dK", K, dK), ("dV", V, dV)]:
    num_grad = np.zeros_like(param)
    it = np.nditer(param, flags=['multi_index'], op_flags=['readwrite'])
    while not it.finished:
        idx = it.multi_index
        orig = param[idx]

        param[idx] = orig + eps
        O_pos, _ = attention_forward(Q, K, V, causal=True)
        loss_pos = np.sum(O_pos * dO)

        param[idx] = orig - eps
        O_neg, _ = attention_forward(Q, K, V, causal=True)
        loss_neg = np.sum(O_neg * dO)

        num_grad[idx] = (loss_pos - loss_neg) / (2 * eps)
        param[idx] = orig
        it.iternext()

    err = rel_error(analytic_grad, num_grad)
    print(f"{name} Relative Error: {err:.2e} | Passed (<1e-6): {err < 1e-6}")

dQ Relative Error: 3.53e-07 | Passed (<1e-6): True
dK Relative Error: 6.46e-08 | Passed (<1e-6): True
dV Relative Error: 1.70e-08 | Passed (<1e-6): True


## Milestone 3: Fraud Detection Evaluation Strategy

### 1. Metrics to Use vs. Avoid
- **Avoid Accuracy & ROC-AUC:** With 0.1% fraud, predicting "all legitimate" gives 99.9% accuracy. ROC-AUC is overly optimistic under extreme class imbalance because massive True Negatives keep the False Positive Rate artificially low.
- **Use PR-AUC (Precision-Recall AUC) & Recall@Fixed-FPR:** PR-AUC focuses strictly on the minority positive class. Also track **Expected Dollar Loss per Transaction**.

### 2. Data Splitting Strategy
- **Temporal + Entity Split:** Split strictly by time (e.g., Train: Months 1–4, Val: Month 5, Test: Month 6) to mimic non-stationary fraud patterns. Group by `card_id` / `user_id` so the same card does not leak across train and test sets.

### 3. Handling 30–60 Day Chargeback Delays
- **Purged Buffer Window:** Leave a 60-day gap before the evaluation window so all test transactions have matured labels.
- **Short-Term Proxy Metrics:** For recent transactions (<30 days), evaluate Precision@K on daily human investigator queues and monitor immediate 3DS authentication failures.

### 4. Cost-Based Decision Threshold & Calibration
- **Costs:** $C_{FN} = \$200$, $C_{FP} = \$5$.
- Flag a transaction as fraud when expected fraud cost exceeds false-alarm cost:
  $$p \cdot C_{FN} \ge (1 - p) \cdot C_{FP} \implies 200p \ge 5(1 - p) \implies 205p \ge 5$$
  $$p^* = \frac{5}{205} \approx 0.0244 \quad (2.44\%)$$
- **Calibration:** Since $p^*$ relies on true probabilities, evaluate calibration via **Brier Score** and **Expected Calibration Error (ECE)** on reliability diagrams. Apply Platt Scaling or Isotonic Regression on the validation set.

### 5. Drift Detection & Retraining Triggers
- **Covariate Drift:** Track **Population Stability Index (PSI)** on input features and output score distributions weekly (alert if $\text{PSI} > 0.2$).
- **Retraining:** Trigger retraining when PSI exceeds threshold, investigator queue precision drops by $>10\%$, or on a rolling bi-weekly schedule using delayed-importance weighting.

In [3]:
# Milestone 3: Cost-Optimal Threshold Verification
c_fn, c_fp = 200.0, 5.0
optimal_threshold = c_fp / (c_fn + c_fp)

# Simulate calibrated probabilities on 100,000 transactions (0.1% fraud)
np.random.seed(42)
y_true = (np.random.rand(100000) < 0.001).astype(int)
probs = np.where(y_true == 1, np.random.beta(2, 3, 100000), np.random.beta(1, 500, 100000))

def total_cost(thresh):
    preds = (probs >= thresh).astype(int)
    fn = np.sum((preds == 0) & (y_true == 1))
    fp = np.sum((preds == 1) & (y_true == 0))
    return fn * c_fn + fp * c_fp

print(f"Theoretical Optimal Threshold: {optimal_threshold:.4f} (2.44%)")
print(f"Total Cost at Default 0.50:    ${total_cost(0.50):,.0f}")
print(f"Total Cost at Optimal 0.0244:  ${total_cost(optimal_threshold):,.0f}")

Theoretical Optimal Threshold: 0.0244 (2.44%)
Total Cost at Default 0.50:    $12,600
Total Cost at Optimal 0.0244:  $0


## Milestone 4: Diagnose Learning Curves

### 1. Diagnosing Cases A, B, and C
- **Case A (Train 62%, Val 60%, flat with more data):** **High Bias (Underfitting).** The model architecture is too simple or lacks predictive features. *Intervention:* Increase model capacity (deeper network/trees) and engineer better domain features.
- **Case B (Train 99%, Val 78%, gap shrinks slowly):** **High Variance (Overfitting).** The model memorizes the training set. *Intervention:* Add regularization (L2 weight decay, Dropout), apply data augmentation, or reduce model complexity.
- **Case C (Train 85%, Val 91%, both rise slightly):** **Train/Val Asymmetry or Heavy Regularization.** Either heavy training-time regularization (aggressive Dropout/Augmentation) is turned off during validation, or the validation split contains artificially easier samples. *Intervention:* Evaluate training accuracy in `eval()` mode (without dropout) and verify stratified splitting.

### 2. Case D (Train 88%, Val 70%, gap stays at 4x data)
- **Two Plausible Explanations:**
  1. **Train/Validation Distribution Mismatch:** Training data comes from a different domain/time period than validation data.
  2. **Systematic Label Noise in Validation Set:** Validation labels have higher noise, creating an artificial accuracy ceiling.
- **Distinguishing Experiment (Data Reshuffle / "Train-Val Swap"):** Pool the training and validation sets together, shuffle randomly, and create a new stratified split. Retrain the model:
  - If the 18% gap **disappears**, the cause was **Distribution Mismatch** between the original splits.
  - If the gap or low ceiling **persists** on specific samples, audit those high-loss validation samples for **Label Noise**.

## 📊 Final Executive Summary

| Milestone | Core Problem | Key Mathematical / Engineering Takeaway |
| :--- | :--- | :--- |
| **1. Pipeline Debugging** | Silent Training Failure | Fixed `X_tr` decoupling shuffle, pre-split `StandardScaler` leak, per-batch LR decay, and missing `model.eval()`. |
| **2. Attention Gradients** | Analytical Backward Pass | Derived $dQ, dK, dV$ and verified in `float64` with finite-difference relative error $< 10^{-10}$. |
| **3. Fraud Evaluation** | Extreme Imbalance (0.1%) | Selected PR-AUC, temporal split with 60-day buffer, and cost-optimal threshold $p^* = 2.44\%$. |
| **4. Learning Curves** | Bias/Variance/Shift | Diagnosed Underfitting (A), Overfitting (B), Eval Regularization (C), and designed Reshuffle Test for Domain Shift (D). |